# e05 — English city-swap robustness (existing checkpoints)

**Реализация:** весь eval-код этого ноутбука в **code cells ниже** (отдельного `.py` для city-swap нет). Общие загрузка модели / батчи — из **`e07_english_eval_common.py`**.

**Данные:** counterfactuals из **32** — `32_english_city_swap_counterfactuals.csv`. Модели — русские чекпоинты, без дообучения на English.

**Выходы:** `notebooks/results/english_city_swap_eval/<model_run_id>/`, фигуры в `figures/english_city_swap_eval/<model_run_id>/`.

**Нужно:** прогнанный **32**, GPU/CPU + `transformers`.


In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import sys
from pathlib import Path
from typing import Any, Dict

import numpy as np
import pandas as pd
import torch

_HERE = Path.cwd().resolve()
if _HERE.name == "english":
    EN_DIR = _HERE
elif (_HERE / "notebooks" / "english" / "e07_english_eval_common.py").is_file():
    EN_DIR = _HERE / "notebooks" / "english"
elif (_HERE / "english" / "e07_english_eval_common.py").is_file():
    EN_DIR = _HERE / "english"
else:
    EN_DIR = _HERE

if str(EN_DIR) not in sys.path:
    sys.path.insert(0, str(EN_DIR))

from e07_english_eval_common import (
    MODEL_CHECKPOINT_CANDIDATES,
    filter_to_encoder_labels,
    load_model_bundle,
    predict_logits_batch,
    resolve_model_dir,
    resolve_repo_root,
)


def _softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def _pair_key(row_id: Any, swap_city: Any, orig: str, swap: str) -> str:
    h = hashlib.sha256(
        f"{row_id}|{swap_city}|{orig[:80]}|{swap[:80]}".encode("utf-8", errors="replace")
    ).hexdigest()[:20]
    return f"ecs_{h}"


def default_city_swap_pairs_path(repo_root: Path) -> Path:
    return (
        repo_root
        / "notebooks"
        / "results"
        / "english_dataset_alignment"
        / "32_english_city_swap_counterfactuals.csv"
    )


def run_english_city_swap_model_eval(
    model_run_id: str,
    *,
    pairs_csv: Path | None = None,
    batch_size: int = 8,
    max_length: int = 256,
    seed: int = 42,
    results_subdir: str = "english_city_swap_eval",
    figures_subdir: str = "english_city_swap_eval",
) -> Dict[str, Any]:
    np.random.seed(seed)
    torch.manual_seed(seed)

    repo_root = resolve_repo_root()
    if pairs_csv is None:
        pairs_csv = default_city_swap_pairs_path(repo_root)
    pairs_csv = Path(pairs_csv)
    if not pairs_csv.is_file():
        raise FileNotFoundError(
            f"Missing English city-swap pairs CSV. Run notebook 32 first: {pairs_csv}"
        )

    model_dir, tried_paths = resolve_model_dir(repo_root, model_run_id)
    results_root = repo_root / "notebooks" / "results" / results_subdir / model_run_id
    figures_root = repo_root / "figures" / figures_subdir / model_run_id
    results_root.mkdir(parents=True, exist_ok=True)
    figures_root.mkdir(parents=True, exist_ok=True)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "mps"
        if torch.backends.mps.is_available()
        else "cpu"
    )

    df = pd.read_csv(pairs_csv)
    need = ["original_text", "swapped_text", "label"]
    for c in need:
        if c not in df.columns:
            raise KeyError(f"City-swap CSV missing {c!r}; got {list(df.columns)}")

    work = df.copy()
    work = work.rename(columns={"label": "supercategory"})
    if "pair_id" not in work.columns:
        rids = work["row_id"].tolist() if "row_id" in work.columns else list(range(len(work)))
        scities = work["swap_city"].tolist() if "swap_city" in work.columns else [""] * len(work)
        work["pair_id"] = [
            _pair_key(r, s, o, sw)
            for r, s, o, sw in zip(
                rids,
                scities,
                work["original_text"].astype(str),
                work["swapped_text"].astype(str),
            )
        ]
    work["anchor_resume_text"] = work["original_text"].fillna("").astype(str)
    work["counterfactual_resume_text"] = work["swapped_text"].fillna("").astype(str)
    if "swap_city" not in work.columns:
        work["swap_city"] = "unknown"
    if "source_city" not in work.columns:
        work["source_city"] = "unknown"

    bundle = load_model_bundle(model_dir, device)
    le = bundle.label_encoder

    work, n_drop = filter_to_encoder_labels(work, "supercategory", le)
    if len(work) == 0:
        raise ValueError("No rows left after filtering to encoder label space")

    y_idx = le.transform(work["supercategory"].astype(str))
    t_a = work["anchor_resume_text"].tolist()
    t_c = work["counterfactual_resume_text"].tolist()

    logits_a = predict_logits_batch(t_a, bundle, device, batch_size=batch_size, max_length=max_length)
    logits_c = predict_logits_batch(t_c, bundle, device, batch_size=batch_size, max_length=max_length)

    pred_a = logits_a.argmax(axis=-1)
    pred_c = logits_c.argmax(axis=-1)
    probs_a = _softmax(logits_a)
    probs_c = _softmax(logits_c)

    n = len(work)
    idx = np.arange(n)
    p_true_a = probs_a[idx, y_idx]
    p_true_c = probs_c[idx, y_idx]
    flip = pred_a != pred_c

    out = work.copy()
    out["y_true_idx"] = y_idx
    out["pred_anchor_idx"] = pred_a
    out["pred_counterfactual_idx"] = pred_c
    out["pred_anchor_label"] = le.inverse_transform(pred_a)
    out["pred_counterfactual_label"] = le.inverse_transform(pred_c)
    out["prob_true_class_anchor"] = p_true_a
    out["prob_true_class_counterfactual"] = p_true_c
    out["prediction_flipped"] = flip.astype(bool)
    out["delta_prob_true_class"] = p_true_c - p_true_a

    pred_path = results_root / "predictions_english_city_swap_pairs.csv"
    out.to_csv(pred_path, index=False)

    flip_rate = float(flip.mean())
    delta_prob_mean = float(out["delta_prob_true_class"].mean())

    by_swap_city = (
        out.groupby("swap_city", dropna=False)
        .agg(
            n_pairs=("pair_id", "count"),
            flip_rate=("prediction_flipped", "mean"),
            mean_delta_prob_true=("delta_prob_true_class", "mean"),
        )
        .reset_index()
    )

    by_true_class = (
        out.groupby("supercategory", dropna=False)
        .agg(
            n_pairs=("pair_id", "count"),
            flip_rate=("prediction_flipped", "mean"),
            mean_delta_prob_true=("delta_prob_true_class", "mean"),
        )
        .reset_index()
    )

    summary: Dict[str, Any] = {
        "eval_kind": "english_city_swap",
        "design_note": "Pairs from 32_english_city_swap_counterfactuals.csv (notebook 32); eval-only stress test on English text.",
        "model_run_id": model_run_id,
        "n_pairs_evaluated": int(len(out)),
        "rows_dropped_unknown_label": int(n_drop),
        "flip_rate": flip_rate,
        "mean_delta_prob_true_class": delta_prob_mean,
        "by_swap_city": by_swap_city.to_dict(orient="records"),
        "by_true_class": by_true_class.to_dict(orient="records"),
        "pairs_csv": str(pairs_csv.relative_to(repo_root)),
        "predictions_csv": str(pred_path.relative_to(repo_root)),
        "resolved_model_dir": str(model_dir),
        "model_resolution_tried": tried_paths,
        "device": str(device),
    }

    (results_root / "metrics.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    manifest = {
        "model_run_id": model_run_id,
        "pid": os.getpid(),
        "results_dir": str(results_root),
        "figures_dir": str(figures_root),
        "metrics_json": str((results_root / "metrics.json").relative_to(repo_root)),
    }
    (results_root / "run_manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(7, 4))
        cities = by_swap_city["swap_city"].astype(str).tolist()
        ax.bar(cities, by_swap_city["flip_rate"].tolist(), color="#72b7b2")
        ax.set_ylabel("Flip rate")
        ax.set_title(f"{model_run_id} — English city-swap (prediction flips by target city)")
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
        fig.tight_layout()
        fp = figures_root / "flip_rate_by_swap_city.png"
        fig.savefig(fp, dpi=160, bbox_inches="tight")
        plt.close(fig)
        summary["figure_flip_rate_by_swap_city"] = str(fp.relative_to(repo_root))
    except Exception as exc:
        summary["figure_error"] = str(exc)

    del bundle, logits_a, logits_c
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


repo = resolve_repo_root()
pairs = default_city_swap_pairs_path(repo)
print("ENGLISH_DIR:", EN_DIR)
print("Pairs CSV:", pairs)
print("Exists:", pairs.is_file(), "| Registered models:", list(MODEL_CHECKPOINT_CANDIDATES.keys()))


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ENGLISH_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/english
Pairs CSV: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/english_dataset_alignment/32_english_city_swap_counterfactuals.csv
Exists: True | Registered models: ['bert_9classes_final', 'bert_scrubbing', 'label_smoothing_eps01_2ep']


In [2]:
MODEL_RUN_ID = os.environ.get("EN_CITYSWAP_MODEL_RUN_ID", "bert_9classes_final")
RUN_ALL = os.environ.get("EN_CITYSWAP_EVAL_ALL", "").strip() in ("1", "true", "True")

ids = list(MODEL_CHECKPOINT_CANDIDATES.keys()) if RUN_ALL else [MODEL_RUN_ID]
summaries = []
for mid in ids:
    print("===", mid, "===")
    try:
        s = run_english_city_swap_model_eval(
            mid,
            pairs_csv=pairs,
            batch_size=8,
            max_length=256,
        )
        summaries.append({k: s.get(k) for k in ("model_run_id", "flip_rate", "mean_delta_prob_true_class", "n_pairs_evaluated")})
    except FileNotFoundError as e:
        print("SKIP:", e)
    except Exception as e:
        print("ERROR:", mid, type(e).__name__, e)

print(json.dumps(summaries, indent=2))


=== bert_9classes_final ===


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1285.81it/s, Materializing param=classifier.weight]                                      


[
  {
    "model_run_id": "bert_9classes_final",
    "flip_rate": 0.03136042402826855,
    "mean_delta_prob_true_class": 5.2644800234702416e-06,
    "n_pairs_evaluated": 6792
  }
]


## Thesis / paper use

- **Complements** Russian **city-swap** (`70`): same models, **English** surface form + geography stress.
- **Differs** from **56–58** minimal-attribute registry: here pairs come from **32**’s alignment pipeline (many swaps per source row / source city).
- Report **overall flip rate**, **mean Δ P(true class)**, and **per-`swap_city` slices** from `metrics.json`.
